# 02 Target and samples

Purpose: confirm the 12-month label and the sample design before modelling.
Every figure here should match `docs/data_quality_log.md`.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import config
from src.splits import assign_sample, sample_summary
from src.eda import missingness_by_year, term_mix, bad_rate_by_term_year, feature_psi, numeric_profile
from src.data_prep import label_summary

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)

df = assign_sample(pd.read_parquet(config.PROCESSED_PATH))
modelling = df[df["sample"] != "excluded"].copy()
print(f"Labelled loans: {len(df):,}  |  in modelling samples: {len(modelling):,}")

## 1. Data quality funnel

In [ ]:
funnel = pd.read_csv(config.FUNNEL_PATH)
display(funnel)

## 2. Label: lifetime outcome against 12-month outcome

In [ ]:
display(label_summary(df))

Only a small share of charge-offs happen inside the first 12 months. This
is the expected gap between a lifetime default rate and a 12-month PD, and
it is why the 12-month definition matters.

## 3. Sample design: size and bad rate

In [ ]:
samples = sample_summary(modelling)
samples["share_of_modelling"] = (samples["loans"] / samples["loans"].sum()).round(3)
display(samples.round(4))

Checks: each sample should have enough bad loans to estimate a stable
default rate. The validation and out-of-time samples should not be
unusually small compared with training.

In [ ]:
bads = modelling.groupby("sample")["target"].sum().rename("bad_loans")
display(bads.to_frame())

## 4. Sample mix by term

In [ ]:
mix_by_sample = pd.crosstab(modelling["sample"], modelling["term"], normalize="index")
display(mix_by_sample.round(3))

If the out-of-time sample has a very different term mix, part of any
performance gap may come from the mix rather than from borrowers. The
validation report should say so.

## 5. Bad rate by sample and term

In [ ]:
by_sample_term = (
    modelling.groupby(["sample", "term"])["target"]
    .agg(loans="size", bad_rate="mean")
    .reset_index()
)
display(by_sample_term.round(4))

Decision point: if the bad rate rises within each term from training to
out-of-time, the shift is real and not only term mix. Record the result
in the validation report.